In [7]:
from collections import defaultdict, deque
from typing import Any

class UserLockTracker:
    def get_locked_user_count(self, logs: list[list[Any]]) -> int:
        """
        Processes login logs and returns the count of currently locked users.
        
        Log Entry Format: [timestamp (int/float), username (str), success_status (bool/str)]
        
        Rules:
        1. 3 failed attempts within a rolling 30-second window locks the user.
        2. A successful login removes the lock and resets failed attempt history.
        """
        if not logs:
            return 0

        # Ensure logs are processed chronologically by timestamp
        sorted_logs = sorted(logs, key=lambda entry: entry[0])

        # State tracking per user
        failed_attempts: dict[str, deque[float]] = defaultdict(deque)
        locked_users: dict[str, bool] = defaultdict(bool)

        print(failed_attempts)
        print(locked_users)

        for timestamp, username, success in sorted_logs:
            # Handle boolean, integer (1/0), or string ("SUCCESS"/"FAIL") indicators
            is_success = success in (True, 1, "SUCCESS", "success", "SUCCESSFUL")

            if is_success:
                # Successful login unlocks the user and resets failure history
                locked_users[username] = False
                failed_attempts[username].clear()
            else:
                user_queue = failed_attempts[username]
                user_queue.append(timestamp)

                # Evict failed attempts older than 30 seconds relative to current timestamp
                while user_queue and (timestamp - user_queue[0]) > 30:
                    user_queue.popleft()

                # Lock user if 3 or more failed attempts occurred within 30 seconds
                if len(user_queue) >= 3:
                    locked_users[username] = True

        # Return count of users who remain locked at the end of stream
        return sum(1 for is_locked in locked_users.values() if is_locked)


# --- Example Test Verification ---
if __name__ == "__main__":
    tracker = UserLockTracker()

    sample_logs = [
        [1000, "alice1", False],
        [1010, "alice1", False],
        [1022, "alice1", False],  # Alice locked (3 fails within 20s)
        
        [1000, "bob", False],
        [1010, "bob", False],
        [1045, "bob", False],    # Bob NOT locked (1000 expired at t=1045)
        
        [1025, "alice", True],   # Alice unlocks via successful login
        [1030, "alice", False],  # Alice fails once after unlock
        
        [1000, "charlie", False],
        [1015, "charlie", False],
        [1028, "charlie", False], # Charlie locked (3 fails within 28s)
    ]

    locked_count = tracker.get_locked_user_count(sample_logs)
    print(f"Total locked users: {locked_count}")  # Output: 1 (Only Charlie remains locked)

defaultdict(<class 'collections.deque'>, {})
defaultdict(<class 'bool'>, {})
Total locked users: 2


In [8]:
# Questions Answered
# Total Active Duration: What is the total active time (in seconds) for each user?

# Abandoned Sessions: How many sessions ended abruptly or timed out (no "END" event and no activity for more than 60 seconds)?

from collections import defaultdict
from typing import Any

class SessionAnalytics:
    def process_sessions(self, logs: list[list[Any]]) -> dict[str, Any]:
        """
        Processes streaming session events and evaluates user session metrics.
        """
        # Ensure events are ordered chronologically
        sorted_logs = sorted(logs, key=lambda x: x[0])

        user_total_time: dict[str, int] = defaultdict(int)
        user_last_start: dict[str, int] = {}
        user_last_seen: dict[str, int] = {}
        timed_out_sessions = 0

        for timestamp, user_id, event in sorted_logs:
            if event == "START":
                # Check if previous session timed out (gap > 60s without END)
                if user_id in user_last_seen and (timestamp - user_last_seen[user_id]) > 60:
                    timed_out_sessions += 1
                
                user_last_start[user_id] = timestamp
                user_last_seen[user_id] = timestamp

            elif event == "HEARTBEAT":
                if user_id in user_last_seen and (timestamp - user_last_seen[user_id]) > 60:
                    timed_out_sessions += 1
                    user_last_start[user_id] = timestamp  # Start new implicit window
                user_last_seen[user_id] = timestamp

            elif event == "END":
                if user_id in user_last_start:
                    duration = timestamp - user_last_start[user_id]
                    user_total_time[user_id] += duration
                    del user_last_start[user_id]
                    del user_last_seen[user_id]

        # Check for unclosed sessions remaining at the end of the log stream
        if sorted_logs:
            last_stream_time = sorted_logs[-1][0]
            for user_id, start_time in list(user_last_start.items()):
                if (last_stream_time - user_last_seen[user_id]) > 60:
                    timed_out_sessions += 1
                else:
                    # Credit partial time up to last active time
                    user_total_time[user_id] += user_last_seen[user_id] - start_time

        return {
            "active_duration_per_user": dict(user_total_time),
            "timed_out_sessions_count": timed_out_sessions
        }

# --- Verification ---
sample_logs = [
    [100, "user_A", "START"],
    [120, "user_A", "HEARTBEAT"],
    [150, "user_A", "END"],          # user_A session 1: 50s

    [100, "user_B", "START"],
    [180, "user_B", "HEARTBEAT"],    # user_B gap = 80s (>60s timeout triggered)
    [200, "user_B", "END"],

    [300, "user_A", "START"],
    [320, "user_A", "HEARTBEAT"],    # user_A session 2 active
]

analyzer = SessionAnalytics()
results = analyzer.process_sessions(sample_logs)
print(results)
# Output:
# {'active_duration_per_user': {'user_A': 70, 'user_B': 20}, 'timed_out_sessions_count': 1}

{'active_duration_per_user': {'user_A': 70, 'user_B': 20}, 'timed_out_sessions_count': 1}
